Circular task analysis

In [10]:
# imports (this is all you need for this assignment - please, do not import anything else)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle , Circle
import os
# very useful for plots that yo can zoom in-out
%matplotlib inline
# if the present working directory is the directory of this file, change the path to the project root
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

In [11]:
data_file = "data/001MoDe_R1.csv"
marker_file = "data/001MoDe_R1.marker.csv"

with open(data_file) as f:
    first_line = f.readline().strip()
header = {}
for item in first_line.split(";"):
    name, value = item.split(" ", 1)
    header[name] = value
print(f"Header line information in '{data_file}':")
for name, value in header.items():
    print(f"  {name}: {value}")

cx, cy = float(header["centerX"]), float(header["centerY"])
Rt = float(header["taskRadius"])
tol = float(header["taskTolerance"])
W, H = float(header["screenWidth"]), float(header["screenHeight"])

with open(marker_file) as f:
    lines = f.read().splitlines()
markers = []
for line in lines[3:]:
    parts = line.split(",", 2)
    if len(parts) == 3 and parts[1].strip().isdigit():
        markers.append((int(parts[1]), parts[2]))
print(f"\nMarkers in '{marker_file}':")
for ts, text in markers:
    print(f"{ts}: {text}")

Header line information in 'data/001MoDe_R1.csv':
  screenWidth: 1104
  screenHeight: 661
  cornerX: 302
  cornerY: 80
  centerX: 552
  centerY: 330
  externalRadius: 250
  internalRadius: 170
  borderRadius: 1
  cursorRadius: 16
  indexOfDifficulty: 28.00696429476858
  taskRadius: 209.5
  taskTolerance: 47
  borderColor: java.awt.Color[r=255,g=255,b=255]
  backgroundColor: java.awt.Color[r=0,g=0,b=0]
  cursorColorRecord: java.awt.Color[r=255,g=0,b=0]
  cursorColorWait: java.awt.Color[r=255,g=255,b=0]
  autoStart: 3600
  cycleMaxNumber: 6
  cycleDuration: 20
  software: LSL-mouse
  version: 1.2.0rc5
  task: CircularTarget
  isWithLSL: false

Markers in 'data/001MoDe_R1.marker.csv':
1616776712325: KeyTyped=32 DoCycleChange
1616776712331: DoCycleChange:DoStartCycleTimedSequence
1616776712333: DoCycleChange:DoRecord RecordDone=0 PauseDone=0 ToDo=6
1616776732336: DoCycleChange:DoPause RecordDone=1 PauseDone=0 ToDo=6
1616776752340: DoCycleChange:DoRecord RecordDone=1 PauseDone=1 ToDo=6
1616

In [12]:
# data: timestamp (ms), mouseX, mouseY, mouseInTarget
d = np.loadtxt(data_file, delimiter=",", skiprows=4)
d = d[d[:, 0] > 0]
t0 = d[0, 0]                                   # time zero = first sample
ts = (d[:, 0] - t0) / 1000                     # time in seconds
x = d[:, 1] - cx                               # origin at the screen center
y = -(d[:, 2] - cy)                            # y axis pointing up
inside = d[:, 3]

# record start = DoRecord marker, record end = next DoPause marker
starts = [m[0] for m in markers if "DoRecord" in m[1]]
pauses = [m[0] for m in markers if "DoPause" in m[1]]
ends = pauses[:len(starts)]
ends += [int(d[-1, 0])] * (len(starts) - len(ends))

print("record times (seconds since the first sample):")
for s, e in zip(starts, ends):
    print(f"{(s - t0)/1000:7.3f} to {(e - t0)/1000:7.3f} ({(e - s)/1000:.3f} s)")

record times (seconds since the first sample):
  0.004 to  20.007 (20.003 s)
 40.011 to  60.022 (20.011 s)
 80.043 to 100.057 (20.014 s)
120.061 to 140.075 (20.014 s)
160.090 to 180.104 (20.014 s)


In [13]:
records = []
for s, e in zip(starts, ends):
    m = (d[:, 0] >= s) & (d[:, 0] <= e)
    records.append(dict(t=ts[m], x=x[m], y=y[m], inside=inside[m]))

print("Number of records:", len(records))
for i, r in enumerate(records):
    dur = r["t"][-1] - r["t"][0]
    print(f"record {i}: {len(r['t'])} samples from {r['t'][0]:7.3f} to {r['t'][-1]:7.3f} s, ({dur:.3f}s, {(len(r['t'])-1)/dur:.3f} Hz)")

Number of records: 5
record 0: 3971 samples from   0.022 to  20.004 s, (19.982s, 198.679 Hz)
record 1: 3898 samples from  40.429 to  60.018 s, (19.589s, 198.938 Hz)
record 2: 3898 samples from  80.371 to 100.050 s, (19.679s, 198.028 Hz)
record 3: 3884 samples from 120.608 to 140.069 s, (19.461s, 199.527 Hz)
record 4: 3936 samples from 160.146 to 180.098 s, (19.952s, 197.223 Hz)


Statistics table